# MIRO AI -- Banc de comparaison de modèles locaux (Colab, gratuit)

**Objectif** : héberger `gpt-oss:20b` et `qwen3:14b` sur le GPU T4 gratuit de Colab et les exposer via un tunnel ngrok. Le banc de synthèse (`scripts/run_synthesis_model_bench.py replay`, `todo/contexte.txt` section 985) tourne sur TA machine et appelle cette URL.

**Confidentialité (ne pas contourner)** : ce notebook ne fait qu'héberger les modèles. Aucune donnée client ni instance Odoo réelle ne doit transiter par Colab, seulement le corpus de test du projet.

**Avant de commencer**
1. `Exécution` -> `Modifier le type d'exécution` -> GPU T4.
2. Secret Colab `NGROK_AUTHTOKEN` (icône clé, "Accès au notebook" activé). Ne jamais coller le token dans une cellule.
3. Garder l'onglet actif (déconnexion après ~90 min d'inactivité, session max ~12 h).
4. En cas d'erreur ou de blocage : exécuter la dernière cellule (diagnostic / redémarrage), pas de relance manuelle en ordre dispersé.

In [ ]:
# 1. GPU assigné
!nvidia-smi

In [ ]:
# 2. Installer Ollama (zstd et pciutils absents de l'image Colab)
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Configuration + stockage des modèles
import os

MODELS = ["gpt-oss:20b", "qwen3:14b"]
CONTEXT_LENGTH = 16384      # baisser (8192) si le modèle déborde sur le CPU (voir `ollama ps`, colonne PROCESSOR)
USE_DRIVE = True            # True : persistant sur Drive ; False : disque local (rapide, ~22 Go re-téléchargés à chaque session)
PULL_EMBED = False          # True seulement pour un run RAG complet
EMBED_MODEL = "nomic-embed-text"

# Attention : 2 modèles = ~22 Go ; un Drive gratuit (15 Go) ne suffit pas. La lecture FUSE est aussi plus lente.
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OLLAMA_MODELS_DIR = "/content/drive/MyDrive/stagiaire@miroingenierie.com/MIRO, MHR, Integration IA Privee/ollama_models"
else:
    OLLAMA_MODELS_DIR = "/content/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)

if USE_DRIVE:
    import shutil, subprocess
    _used = int(subprocess.run(["du", "-sb", OLLAMA_MODELS_DIR], capture_output=True, text=True).stdout.split()[0])
    _free = shutil.disk_usage("/content/drive/MyDrive").free
    print(f"Drive : {_used / 1e9:.1f} Go déjà utilisés par les modèles, {_free / 1e9:.1f} Go libres (besoin total ~22 Go).")
    if _used + _free < 23e9:
        print("ATTENTION : espace Drive insuffisant pour les 2 modèles -> le pull échouera. Libérer de l'espace ou USE_DRIVE = False.")

os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
# Désactive le contrôle CORS Origin. Le rejet 403 du header Host est traité côté ngrok (host_header).
os.environ["OLLAMA_ORIGINS"] = "*"
# Fenêtre par défaut = 4096 tokens (prompt + réponse) : coupe la synthèse multi-hop (Section 710).
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LENGTH)
# Ne jamais décharger un modèle entre deux appels...
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
# ...mais un seul à la fois : les 2 modèles ne tiennent pas ensemble dans ~15 Go de VRAM.
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
# Délai max de chargement d'un modèle (défaut 5 min) : la lecture mmap depuis Drive (FUSE) le dépasse
# ("timed out waiting for llama-server to start" -> HTTP 500).
os.environ["OLLAMA_LOAD_TIMEOUT"] = "30m"
print(f"Modèles stockés dans : {OLLAMA_MODELS_DIR}")

In [ ]:
# 4. Serveur Ollama (rejouable : tue tout puis redémarre)
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"
LOG_PATH = "/content/ollama_serve.log"
ollama_proc = None

def stop_ollama():
    # -x : nom exact du process (évite que pkill se tue lui-même via `-f`) ; tue aussi les runners
    subprocess.run(["pkill", "-9", "-x", "ollama"])
    time.sleep(3)

def log_tail(n=40):
    try:
        return "".join(open(LOG_PATH, errors="replace").readlines()[-n:])
    except FileNotFoundError:
        return "(pas de log)"

def start_ollama():
    global ollama_proc
    stop_ollama()
    # Log dans un FICHIER, pas dans un pipe : un pipe jamais lu se remplit et bloque le serveur.
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open(LOG_PATH, "w"),
                                   stderr=subprocess.STDOUT, env=os.environ)
    for _ in range(60):
        try:
            if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
                print("Ollama prêt.")
                return
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama n'a pas démarré en 60 s. Log :\n" + log_tail())

def ollama_ps():
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    return r.stdout + r.stderr

def diag():
    print("process :", "vivant" if ollama_proc and ollama_proc.poll() is None else f"mort (code {ollama_proc.poll() if ollama_proc else '?'})")
    try:
        requests.get(OLLAMA_URL, timeout=3)
        print("port 11434 : répond")
    except requests.exceptions.RequestException:
        print("port 11434 : ne répond pas")
    print(ollama_ps())
    print("--- fin du log serveur ---\n" + log_tail())

start_ollama()
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 5. Télécharger les modèles
for _m in MODELS + ([EMBED_MODEL] if PULL_EMBED else []):
    !ollama pull {_m}
!ollama list

In [ ]:
# 6. Chargement + test de tool-calling, un modèle après l'autre
# Le payload inclut 'tools' : Ollama instancie un runner différent selon leur présence (Section 689).
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Renvoie la météo pour une ville donnée.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
    },
}

def load_and_test(model, timeout=900):
    t0 = time.time()
    try:
        r = requests.post(f"{OLLAMA_URL}/v1/chat/completions", timeout=timeout, json={
            "model": model, "stream": False, "tools": [WEATHER_TOOL],
            "messages": [{"role": "user", "content": "Quel temps fait-il à Paris ?"}],
        })
        r.raise_for_status()
        calls = r.json()["choices"][0]["message"].get("tool_calls")
        print(f"{model} : OK en {time.time() - t0:.0f} s, tool-calling {'OK' if calls else 'absent (non bloquant pour le banc de synthèse)'}")
    except Exception as e:
        print(f"{model} : ÉCHEC {type(e).__name__} après {time.time() - t0:.0f} s")
        diag()
        raise
    print(ollama_ps())  # PROCESSOR doit afficher 100% GPU

for _m in MODELS:
    load_and_test(_m)

## 7. Tunnel ngrok

Secret Colab `NGROK_AUTHTOKEN` requis (voir en haut).

In [ ]:
# 7. Exposer Ollama via ngrok
!pip install -q pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata

conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")

# host_header : Ollama rejette en 403 tout Host différent de localhost (Sections 682, 694).
for _t in ngrok.get_tunnels():
    ngrok.disconnect(_t.public_url)
tunnel = ngrok.connect(11434, "http", host_header="localhost:11434")
PUBLIC_URL = tunnel.public_url

r = requests.get(f"{PUBLIC_URL}/api/tags", timeout=30)
print("Test via le tunnel : HTTP", r.status_code, "-", [m["name"] for m in r.json().get("models", [])])
print("\nURL publique (valable tant que ce notebook tourne) :")
print(PUBLIC_URL)

## 8. Côté de ta machine (pipeline MIRO AI + accès Odoo)

### 8a. Banc de comparaison de synthèse

**Étape 1, `capture`** (une seule fois, depuis ta machine, sans que ce notebook soit lancé : elle utilise ton LLM habituel et l'instance Odoo, et enregistre les prompts dans `captures/frozen_evidence/`) :

```bash
python scripts/run_synthesis_model_bench.py capture MNT-Q1 SAL-Q1 ACC-Q1 STK-Q1 PUR-Q1 PUR-Q5
```

**Étape 2, `replay`** (notebook actif, URL affichée à la cellule 7). Un modèle à la fois : le premier appel sert de préchauffage, ce qui évite un timeout au changement de modèle.

```bash
python scripts/run_synthesis_model_bench.py replay --base-url <URL_NGROK>/v1 --model ollama:gpt-oss:20b --samples 3 --report-stem captures/bench_gptoss
python scripts/run_synthesis_model_bench.py replay --base-url <URL_NGROK>/v1 --model ollama:qwen3:14b  --samples 3 --report-stem captures/bench_qwen3
```

`LOCAL_LLM_API_KEY=ollama` si le client le réclame. Le notebook reçoit les prompts de synthèse gelés (données de l'instance de TEST, chemins absolus de ta machine) : aucune donnée client réelle dans l'instance utilisée pour le `capture`.

### 8b. Autre usage : banc adversarial sur un modèle local

Dans le `.env` (temporairement) :

```
LLM_PROVIDER=ollama
LOCAL_LLM_BASE_URL=<URL_NGROK>/v1
LOCAL_LLM_MODEL=gpt-oss:20b   # ou qwen3:14b
LOCAL_LLM_API_KEY=ollama
```

```bash
python scripts/run_adversarial_benchmark.py
```

Rapport : `todo/verbatim_responses_sprint1.md` / `.json`. **Remettre `LLM_PROVIDER=groq` (ou retirer ces 4 lignes) après le test.**

**Limite Colab gratuit** : session ~12 h max, déconnexion après ~90 min d'inactivité (Colab Pro en secours, `todo/contexte.txt` Section 674).

In [ ]:
# En cas de blocage / timeout : diagnostic, puis (si le serveur est mort ou bloqué) redémarrage.
diag()
# start_ollama()   # décommenter pour redémarrer ; ensuite relancer la cellule 6 (rechargement des modèles)